# Distance and reservoir-size models: results (Stages 3-4)

How the dam-removal effect on each water-quality measure changes with distance downstream, and whether it is larger for larger reservoirs. Reads the outputs of:

1. `sbatch data/slurm/fit_distance_models.slurm` (random effects `auto`), plus `RE=intercepts` and `RE=slopes` with `RUNS="primary size_proxy"` (comparisons) -> `data/processed/mixed_model/<case>/stage3/<run>__<re>/`
2. `sbatch data/slurm/fit_stage4.slurm` -> `stage4/influence/` and `stage4/report/`

The notebook fits nothing; if the report tables are missing it rebuilds them from the model outputs (seconds).

**Reading the results**
- **BACI** (downstream change minus upstream-control change at the same dam) is the primary evidence; **before/after** (downstream only, no control for regional trends) is supplementary.
- Each estimate is classified: **detected** (95% CI excludes 0), **no meaningful change** (the *entire* 95% CI inside +/- the smallest effect of interest, SESOI: a small, *precise* estimate), or **inconclusive** (the data can't tell a meaningful effect from none). The SESOI values are **analyst-selected practical-significance thresholds, not environmental or regulatory standards** (section 9 lists them and shows which classifications change under smaller or larger thresholds); they are set in `analysis/stage4_report.py`.
- **MDE** (minimum detectable effect, 80% power): the true effect the model would detect 80% of the time. An inconclusive result means "smaller than about the MDE, or none".
- Dams, not measurements, are the independent units (intervals use dams - 1 degrees of freedom). Fits with fewer than 15 dams are flagged exploratory.
- These are observational associations, not proof that a removal caused a change.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import mixed_model as mm
from analysis import stage4_report as rep

CASE_STUDY = "all_removals"
# DAM_REMOVAL_MODEL_DIR points at a copy of mixed_model/<case> (e.g. downloaded outputs).
MODEL_DIR = Path(os.environ.get("DAM_REMOVAL_MODEL_DIR", mm.output_dir_for(CASE_STUDY, ROOT)))
REPORT_DIR = MODEL_DIR / "stage4" / "report"
INFLUENCE_DIR = MODEL_DIR / "stage4" / "influence"

if not (REPORT_DIR / "model_summary.csv").exists():
    rep.build(MODEL_DIR, REPORT_DIR, figures=False)
read = lambda name: pd.read_csv(REPORT_DIR / f"{name}.csv") if (REPORT_DIR / f"{name}.csv").stat().st_size > 1 else pd.DataFrame()
summary, by_distance, curve = read("model_summary"), read("effects_by_distance"), read("effects_curve")
sensitivity, comparison, influence = read("sensitivity_table"), read("distance_comparison"), read("influence_summary")
thresholds, threshold_sensitivity = read("sesoi_thresholds"), read("sesoi_sensitivity")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
primary = summary[(summary["run"] == rep.PRIMARY[0]) & (summary["re"] == rep.PRIMARY[1])]
print(f"Model outputs: {MODEL_DIR}")
print(f"Runs: {sorted(set(summary['run'] + '__' + summary['re']))}")

## 1. Primary results: effect at 5 km and how it changes with distance

`effect_at_reference` is the effect 5 km below the dam (drainage-area ratio at the fit's median); `modifier_slope` is the change per unit of ln(1 + km). Series with too few dams for a model are listed with the reason.

In [ ]:
cols = ["series", "design", "dams", "stations", "days", "sample_flag", "re_choice",
        "effect_at_reference", "effect_ci_low", "effect_ci_high", "mde80_effect", "sesoi", "effect_class",
        "modifier_slope", "modifier_ci_low", "modifier_ci_high", "slope_class", "error"]
for spec in ["baci_distance", "before_after_distance"]:
    print(f"\n{spec}")
    display(primary[primary["spec"] == spec][cols])

## 2. Distance-response curves

Solid line and band: the primary model (dam slopes where supported) with its 95% CI, inside each fit's observed distance range only. Dashed: the intercept-only fit (sensitivity). Points: the reported distances. Grey band: +/- the smallest effect of interest. Log-scale series are in log10 units (0.1 = +26%).

In [ ]:
for series in dict.fromkeys(curve["series"]):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    rep.plot_distance_response(curve, by_distance, series, axes, rep.SESOI.get(series))
    fig.tight_layout()
    plt.show()

### Effects at representative distances (primary)

Percent change for log-scale series. Distances outside a fit's observed range are not reported.

In [ ]:
points = by_distance[by_distance["primary"] & by_distance["distance_km"].notna()
                    & by_distance["spec"].isin(["baci_distance", "before_after_distance"])]
show = ["series", "design", "distance_km", "estimate", "ci_low", "ci_high", "mde80", "effect_class",
        "pct_estimate", "pct_ci_low", "pct_ci_high", "dams", "sample_flag"]
display(points[[c for c in show if c in points]].sort_values(["series", "design", "distance_km"]))

## 3. Sensitivity: does the conclusion depend on choices we made?

Rows: primary; primary with intercept-only random effects; drainage-area-ratio limits 100 / 10 / 2 (only stations where the dam supplies at least 1% / 10% / 50% of the water); suspect dam matches kept in.

In [ ]:
for km in (1, 5, 20):
    table = sensitivity[sensitivity["distance_km"] == km].copy()
    if table.empty:
        continue
    table["cell"] = (table["estimate"].map("{:+.3f}".format) + " [" + table["ci_low"].map("{:+.3f}".format)
                     + ", " + table["ci_high"].map("{:+.3f}".format) + "] n=" + table["dams"].astype(int).astype(str))
    print(f"\nEffect at {km} km")
    display(table.pivot_table(index=["series", "design"], columns="model_run", values="cell", aggfunc="first"))
for design in ("BACI", "before/after"):
    if len(sensitivity):
        rep.plot_sensitivity(sensitivity, design)
        plt.show()

## 4. Random effects: dam slopes or intercepts?

`auto` keeps dam random slopes (each dam its own effect) only where the data support them: the effect's dam slope variance isn't ~0 and a likelihood-ratio test of slopes vs intercepts gives p < 0.05. Otherwise intercepts.

The test is on the boundary (a variance can't be below 0), so the plain chi-square p (`lrt_p_chi2`) is only an upper bound: when it is below 0.05 that settles it; otherwise `lrt_p` comes from a parametric bootstrap (data simulated from the intercepts model, both models refit). `lrt_p_method` says which. The width ratios show how much assuming one effect for every dam narrows the interval, against intercept-only and forced-slopes fits.

In [ ]:
auto = summary[(summary["run"] == "primary") & (summary["re"] == "auto")]
width = lambda frame: frame["effect_ci_high"] - frame["effect_ci_low"]
table = auto.copy()
for other in ("intercepts", "slopes"):
    alt = summary[(summary["run"] == "primary") & (summary["re"] == other)][["series", "spec", "effect_ci_low", "effect_ci_high"]]
    if len(alt):
        alt = alt.assign(**{f"width_{other}": width(alt)})[["series", "spec", f"width_{other}"]]
        table = table.merge(alt, on=["series", "spec"], how="left")
        table[f"ci_width_auto_vs_{other}"] = width(table) / table[f"width_{other}"]
cols = ["series", "spec", "dams", "re_choice", "lrt_stat", "lrt_df", "lrt_p_chi2", "lrt_p", "lrt_p_method",
        "singular", "near_zero_terms", "fit_check", "ci_width_auto_vs_intercepts", "ci_width_auto_vs_slopes"]
display(table[[c for c in cols if c in table]])

## 5. Influence of individual dams (primary BACI distance models)

Leave-one-dam-out refits (random-effects structure and reference values held fixed): a dam is *influential* when leaving it out moves the effect at 5 km or the distance slope by at least one standard error, or changes whether the interval excludes zero. The dam bootstrap interval is next to the model interval: if they disagree, prefer the wider one.

In [ ]:
if influence.empty:
    print("No influence outputs yet (run data/slurm/fit_stage4.slurm).")
else:
    cols = [c for c in ["series", "spec", "dams", "full_effect_at_reference", "full_effect_ci_low",
                        "full_effect_ci_high", "lodo_effect_min", "lodo_effect_max", "influential_dams",
                        "influential_dam_ids", "effect_conclusion_changes", "slope_conclusion_changes",
                        "draws_failed", "effect_at_reference_boot_low", "effect_at_reference_boot_high",
                        "modifier_slope_boot_low", "modifier_slope_boot_high"] if c in influence]
    display(influence[cols])
lodo_path = INFLUENCE_DIR / "lodo.csv"
if lodo_path.exists():
    lodo = pd.read_csv(lodo_path)
    series_list = list(dict.fromkeys(lodo["series"]))
    fig, axes = plt.subplots(len(series_list), 1, figsize=(12, 2.2 * len(series_list)), squeeze=False)
    for ax, s in zip(axes[:, 0], series_list):
        g = lodo[lodo["series"] == s].sort_values("delta_effect_at_reference_in_se")
        colors = np.where(g["influential"].fillna(False).astype(bool), "C3", "C0")
        ax.bar(g["dam_left_out"].astype(str), g["delta_effect_at_reference_in_se"], color=colors)
        ax.axhline(1, ls=":", color="grey"); ax.axhline(-1, ls=":", color="grey")
        ax.set_title(f"{s}: change in the 5 km effect when each dam is left out (full-fit SEs; red = influential)", fontsize=9)
        ax.tick_params(axis="x", labelrotation=90, labelsize=6)
    fig.tight_layout()
    plt.show()

## 6. Raw vs reservoir-length-scaled distance (same dams and days)

Both specifications are fit on exactly the same rows (the dams with a reservoir-length estimate), with the same covariates apart from the distance term, by maximum likelihood with intercept-only random effects, so the AIC difference compares only how distance is measured (`same_sample`, `same_covariates`, `same_random_effects` and `both_ml` confirm it). |delta AIC| < 2: no clear difference. A focused comparison of two distance formulations, not a model search.

In [ ]:
display(comparison if len(comparison) else pd.DataFrame({"note": ["no common_sample run yet"]}))

## 7. pH: is the positive BACI effect a downstream rise, an upstream fall, or both?

The BACI effect is (downstream change) - (upstream change). Below: each dam's average change after removal at its upstream controls and at its downstream stations, the model's own split, and the dam-averaged yearly anomaly (value minus the station's pre-removal mean) for each role. Seasonal sampling differences between years are not removed from the trajectory, so read it as a description. Also shown for water temperature and dissolved oxygen for comparison.

In [ ]:
comp_path = INFLUENCE_DIR / "components_summary.csv"
if comp_path.exists():
    display(pd.read_csv(comp_path))
for series in ["ph", "water_temp", "dissolved_oxygen"]:
    traj_path, comp_series_path = INFLUENCE_DIR / f"trajectory_{series}.csv", INFLUENCE_DIR / f"components_{series}.csv"
    if not traj_path.exists():
        continue
    traj, comp = pd.read_csv(traj_path), pd.read_csv(comp_series_path)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for role, color in (("control", "C1"), ("treated", "C0")):
        g = traj[traj["role"] == role].sort_values("years_from_removal")
        label = "upstream controls" if role == "control" else "downstream"
        axes[0].errorbar(g["years_from_removal"], g["mean"], yerr=1.96 * g["se"], color=color,
                         marker="o", capsize=2, label=label)
    axes[0].axvline(0, color="black", lw=0.8); axes[0].axhline(0, color="grey", lw=0.6)
    axes[0].set(xlabel="years from removal", ylabel="anomaly vs station's pre-removal mean",
                title=f"{series}: dam-averaged trajectory")
    axes[0].legend()
    axes[1].scatter(comp["change_upstream"], comp["change_downstream"])
    lim = np.nanmax(np.abs(comp[["change_upstream", "change_downstream"]].to_numpy())) * 1.1
    axes[1].plot([-lim, lim], [-lim, lim], color="grey", lw=0.8, ls="--")
    axes[1].axhline(0, color="grey", lw=0.6); axes[1].axvline(0, color="grey", lw=0.6)
    axes[1].set(xlabel="change after removal, upstream controls", ylabel="change after removal, downstream",
                title=f"{series}: one point per dam (above the dashed line = positive BACI)")
    fig.tight_layout()
    plt.show()

## 8. Reservoir size (exploratory)

Does the effect differ for larger reservoirs, holding river size (dam drainage area) fixed? `modifier_slope` is the change in the effect per standard deviation of log10 reservoir size. Few BACI dams have both measures, so read these as exploratory. `size_vs_drainage_vif` checks that size and river size aren't too collinear.

In [ ]:
size = summary[summary["run"].isin(["size_proxy", "size_polygon"])]
display(size[["run", "re", "series", "spec", "dams", "dams_before_modifier", "re_choice", "modifier_slope",
              "modifier_ci_low", "modifier_ci_high", "mde80_modifier", "slope_class", "size_vs_drainage_r",
              "size_vs_drainage_vif", "sample_flag", "error"]])

## 9. Thresholds, detectable effects, and how much the labels depend on the thresholds

The smallest effects of interest below are **analyst-selected practical-significance thresholds (provisional), not environmental or regulatory standards**. "No meaningful change" requires the entire 95% CI inside +/- the threshold. Where the MDE is larger than the threshold, an inconclusive result is expected even if a meaningful effect exists: the data are underpowered, not evidence of no effect. The last table re-classifies every primary estimate at 0.5x-2x the thresholds; rows where the label changes are the ones whose wording depends on the threshold choice.

In [ ]:
display(thresholds)
mde = primary[primary["error"].fillna("") == ""][["series", "spec", "dams", "mde80_effect", "sesoi", "sesoi_label", "effect_class"]].copy()
mde["powered_for_threshold"] = mde["mde80_effect"] <= mde["sesoi"]
display(mde)
if len(threshold_sensitivity):
    changed = threshold_sensitivity[threshold_sensitivity["class_changes"]]
    print(f"{len(changed)} of {len(threshold_sensitivity)} primary estimates change label between 0.5x and 2x the thresholds:")
    display(changed.drop(columns=["thresholds"]))

## Notes

- Distance is network distance along the river; the drainage-area ratio (station / dam NHDPlus drainage area) is a dilution proxy, not a measured flow.
- The 12 suspected dam-matching errors (`config/suspect_dam_matches.csv`) are excluded from every run except `keep_suspects`.
- The reservoir-volume proxy (height^2 / slope x sqrt(drainage area)) is a constructed size index, not a measured volume or stored sediment.
- Tables for the report and Tableau: `stage4/report/*.csv`; figures: `stage4/report/figures/`.